<a href="https://colab.research.google.com/github/danielajetunmobi/flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: classification feeding a ranking.**

Classification means teaching a model to answer a yes/no question. Here, three separate yes/no questions, each its own model:
- "Will this page decline?"
- "Will this page recover?"
- "Will this page gain momentum?"

They're split into three because each question only makes sense for a different starting group of pages — "recover" only applies to pages already struggling, "momentum" only applies to pages that were fine. Cramming all three into one label would blur cases that mean opposite things.

But a model doesn't just answer "yes" or "no" — it gives a confidence number, like "78% likely yes" (a **probability**). That's where the ranking comes in: once every page has one of these percentages, sort all of them by that number, biggest first. That sorted list *is* the review queue a specialist works through — the page scored 0.91 sits at the top and gets looked at first; the page scored 0.08 sits near the bottom.

So "classification feeding a ranking" means the *same* confidence number does two jobs at once: it's the answer to the yes/no question, and it's the sort key that decides review order. This is the same shape as the reference pipeline — `scripts/03_train_model.py` trains a classifier, and `04_evaluate_and_export.py` turns its probability into the ranked list.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Grain:** one row = one (content item, decision-point date). This needs the warehouse's daily fact table (`fact_content_daily_performance`) — the starter CSV has no daily granularity, so it can't supply a real *future* window at all.

**Windows (for a fixed decision point, e.g. 2026-03-31):**
- **Trend check (30 vs 30 days):** whether a page was *already* declining coming in uses the most recent 60 days, split into two 30-day halves — this matches FlyRank's own `trend_pct` convention exactly (`(last_30d - prev_30d) / prev_30d`, see `docs/data-dictionary.md`), rather than a window we invented. The sanity-check cell tests this against a wider 45-vs-45 split of the full 90 days, and the two disagree on **21.9% of pages** — since this flag decides which of the three target populations a page belongs to, matching the established convention rather than guessing mattered.
- **Future window (30 days):** the 30 days after the decision point, compared against the **most recent 30 days** (not a 90-day average — see the baseline sanity-check cell). A 90-day average can carry an older, no-longer-representative period (e.g. healthy for 60 days, then declining for 30), which hides real recoveries against a stale blend. Switching to the recent-30-day baseline moved `future_recovery` from 18.3% to 28.2% — a real effect, not a rounding difference.
- A 90-day total is still pulled from the query, but only to reconstruct that rejected 90-day baseline for the sanity check with real numbers — it isn't a model feature right now (dropped, since nothing currently uses it).

**The three targets, all observed future outcomes (not a rule applied to the current window):**
- `future_decline` = page was NOT already declining, AND `future_change_pct <= -20%`
- `future_recovery` = page WAS already declining, AND `future_change_pct >= +20%`
- `future_momentum` = page was NOT already declining, AND `future_change_pct >= +20%`

None of these come from someone's existing rule — they're all computed from what actually happened in the 30 days *after* the decision point, which is exactly what keeps this an observed label rather than a proxy for a decision someone already made.

**Open question for ML-04/05:** switching to a 30-day baseline fixed the stale-history problem, but it also made `future_decline` and `future_momentum` shift a lot more than just `future_recovery` did (see the base rates below) — a 30-day window is noisier than a 90-day one, so some of that shift is probably short-term volatility in the baseline itself, not all of it "fixing a bug." Worth testing a minimum-volume filter or a slightly wider recent window (e.g. 45 days) against this before finalizing. This is also where an actual volume feature (page size, currently dropped) would likely need to come back in — probably as its own separate feature, not folded into the trend/baseline math.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: Precision@50** on the ranked queue — matches a specialist's realistic weekly review capacity, same metric the reference pipeline reports (baseline 0.240 → random forest 0.740 on the decline label).

**Reported alongside it, not instead of it:** Recall@50 and the base rate for each target. From `w01`'s cost analysis, a missed real decline/recovery costs more than a false flag reviewed for nothing — so precision alone isn't enough; if recall is very low, the model is quietly missing exactly the cases that matter most, even while looking precise on its top 50.

**"Good" means:** all three models beat their own base rate by a clear margin (not just edge out chance), and Precision@50 clears roughly double the base rate — mirroring the ~3x lift the reference pipeline already demonstrates on its decline label.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib python-dotenv

import os
import duckdb
import pandas as pd
from huggingface_hub import hf_hub_download

# Token: prefer an env var from a local, never-committed .env; otherwise prompt.
# Never hardcode a token in a cell -- this repo is public (see SETUP.md).
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

token = os.environ.get("HF_TOKEN")
if not token:
    import getpass
    token = getpass.getpass("Hugging Face token (read): ")

# Only the 5 month-partitions this decision point's windows actually touch --
# iterate on a slice, never scan the full ~79M-row table for a feasibility check.
# Downloading the whole (small) partition files is more reliable here than
# streaming range-reads over hf:// -- fewer moving parts, resumable on failure.
MONTHS = ["2025-12", "2026-01", "2026-02", "2026-03", "2026-04"]
local_files = [
    hf_hub_download(
        repo_id="FlyRank/internship-warehouse",
        repo_type="dataset",
        filename=f"fact_content_daily_performance/month={m}/data_0.parquet",
        token=token,
    )
    for m in MONTHS
]

con = duckdb.connect()
file_list = ", ".join(f"'{f}'" for f in local_files)
REL = f"read_parquet([{file_list}])"

DECISION_DATE = "2026-03-31"  # fixed decision point for this feasibility check

query = f"""
WITH prior AS (
    SELECT content_hash_id,
           -- full 90-day total: NOT a model feature (dropped -- currently
           -- unused). Kept only so the rejected-baseline sanity check below
           -- can reconstruct and show real numbers for what we tested against.
           SUM(gsc_impressions) AS prior_impressions_90d_for_rejected_baseline_test,
           -- the 30/30 window: trend check AND the future-window baseline both
           -- use this cadence (matches FlyRank's own trend_pct convention).
           SUM(gsc_impressions) FILTER (
               WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 60 DAY
                 AND report_date < DATE '{DECISION_DATE}' - INTERVAL 30 DAY
           ) AS trend_baseline_impr,
           SUM(gsc_impressions) FILTER (
               WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 30 DAY
           ) AS trend_recent_impr,
           -- the wider 45/45 window, kept ONLY for the window-choice sanity
           -- check below -- never used in the actual label definitions.
           SUM(gsc_impressions) FILTER (
               WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 90 DAY
                 AND report_date < DATE '{DECISION_DATE}' - INTERVAL 45 DAY
           ) AS prior_first_half_impr,
           SUM(gsc_impressions) FILTER (
               WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 45 DAY
           ) AS prior_second_half_impr
    FROM {REL}
    WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 90 DAY
      AND report_date < DATE '{DECISION_DATE}'
    GROUP BY content_hash_id
),
future AS (
    SELECT content_hash_id, SUM(gsc_impressions) AS future_impressions
    FROM {REL}
    WHERE report_date >= DATE '{DECISION_DATE}'
      AND report_date <= DATE '{DECISION_DATE}' + INTERVAL 30 DAY
    GROUP BY content_hash_id
)
SELECT p.*, f.future_impressions
FROM prior p JOIN future f USING (content_hash_id)
-- trend_recent_impr > 0 is required for everyone, since it's the denominator
-- for both was_declining AND future_change_pct below.
WHERE p.prior_impressions_90d_for_rejected_baseline_test > 0
  AND p.trend_baseline_impr > 0 AND p.trend_recent_impr > 0
"""

df = con.sql(query).df()
print("Grain: one row = one (content item, decision-point date =", DECISION_DATE, ")")
print("Rows (content items with data in all three windows):", len(df))
df.head()

Note: you may need to restart the kernel to use updated packages.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Grain: one row = one (content item, decision-point date = 2026-03-31 )
Rows (content items with data in all three windows): 134398


,content_hash_id,prior_impressions_90d_for_rejected_baseline_test,trend_baseline_impr,trend_recent_impr,prior_first_half_impr,prior_second_half_impr,future_impressions
0,content_2b94b932002c4b69,61.0,19.0,42.0,NaN,61.0,30.0
1,content_20c2722ee1327431,226.0,20.0,206.0,NaN,226.0,155.0
2,content_b296e548ed6cdcaa,1600.0,119.0,1481.0,NaN,1600.0,1400.0
3,content_0ab0648f7cd969b9,63.0,8.0,55.0,NaN,63.0,22.0
4,content_e0ed7e45dd36727d,1871.0,329.0,1542.0,NaN,1871.0,197.0


**Window-choice sanity check.** Before picking the 30/30 window for `was_declining`, I tested it against a wider 45/45 split of the same 90 days -- this is that actual test, not just a claim.

In [2]:
# This sanity check gets its own local subset (needs prior_first_half_impr > 0
# to divide safely) -- it must NOT filter or otherwise change `df`, which is the
# actual population every other cell in this notebook builds on.
check = df[df["prior_first_half_impr"] > 0].copy()

check["declining_30_30"] = ((check["trend_recent_impr"] - check["trend_baseline_impr"]) / check["trend_baseline_impr"] * 100) <= -20
check["declining_45_45"] = ((check["prior_second_half_impr"] - check["prior_first_half_impr"]) / check["prior_first_half_impr"] * 100) <= -20

agreement = (check["declining_30_30"] == check["declining_45_45"]).mean()
print("Sanity-check subset (needs data in the oldest third of the 90 days too): n =", len(check))
print("declining rate, 30/30 (FlyRank's own trend_pct convention):", round(check["declining_30_30"].mean(), 3))
print("declining rate, 45/45 (a wider split of the full 90 days): ", round(check["declining_45_45"].mean(), 3))
print()
print("agreement between the two window choices:", round(agreement, 3))
print("-> the two definitions disagree on", round((1 - agreement) * 100, 1),
      "% of pages -- window choice isn't cosmetic, it decides which of the",
      "three target populations a page falls into. That's why the model below",
      "uses the 30/30 window: it matches FlyRank's own established convention",
      "rather than a width we picked ourselves.")

Sanity-check subset (needs data in the oldest third of the 90 days too): n = 106930
declining rate, 30/30 (FlyRank's own trend_pct convention): 0.264
declining rate, 45/45 (a wider split of the full 90 days):  0.192

agreement between the two window choices: 0.777
-> the two definitions disagree on 22.3 % of pages -- window choice isn't cosmetic, it decides which of the three target populations a page falls into. That's why the model below uses the 30/30 window: it matches FlyRank's own established convention rather than a width we picked ourselves.


In [3]:
# Prior trend: last 30 days vs the 30 days before that, +/-20% convention --
# matches FlyRank's own trend_pct exactly (see data-dictionary.md), rather than
# an arbitrary window we invented ourselves.
df["prior_trend_pct"] = (
    (df["trend_recent_impr"] - df["trend_baseline_impr"]) / df["trend_baseline_impr"] * 100
)
df["was_declining"] = df["prior_trend_pct"] <= -20

# Future change: recent 30-day daily rate vs future 30-day daily rate.
# NOT the 90-day average -- a page's 90-day average can still be inflated by
# an older, no-longer-representative period (e.g. healthy for 60 days, then
# declining for 30), which would hide a real recovery against a stale blend.
# Comparing against the same recent 30 days used for the trend check is the
# fair, apples-to-apples baseline (verified below: this changed the
# future_recovery rate from 18.3% to 28.2% -- not a cosmetic difference).
recent_daily = df["trend_recent_impr"] / 30
future_daily = df["future_impressions"] / 30
df["future_change_pct"] = (future_daily - recent_daily) / recent_daily * 100

df["future_decline"] = (~df["was_declining"]) & (df["future_change_pct"] <= -20)
df["future_recovery"] = (df["was_declining"]) & (df["future_change_pct"] >= 20)
df["future_momentum"] = (~df["was_declining"]) & (df["future_change_pct"] >= 20)

# Diagnostic only -- not a fourth model target. This is the implicit negative class
# for all three targets above: confirms "no flag fires" actually means "didn't move
# much," rather than silently swallowing a real case the three targets miss.
df["stagnant"] = (df["future_change_pct"] > -20) & (df["future_change_pct"] < 20)

print("Base rates:")
print("  was_declining (prior state):", round(df["was_declining"].mean(), 3))
print("  future_decline (of all pages):", round(df["future_decline"].mean(), 3))
print(
    "  future_recovery (of already-declining pages, n=%d):" % df["was_declining"].sum(),
    round(df.loc[df["was_declining"], "future_recovery"].mean(), 3),
)
print(
    "  future_momentum (of non-declining pages, n=%d):" % (~df["was_declining"]).sum(),
    round(df.loc[~df["was_declining"], "future_momentum"].mean(), 3),
)
print()
print("Diagnostic -- stagnant (implicit negative class), split by prior state:")
print(
    "  stagnant, of non-declining pages:",
    round(df.loc[~df["was_declining"], "stagnant"].mean(), 3),
)
print(
    "  stagnant, of already-declining pages:",
    round(df.loc[df["was_declining"], "stagnant"].mean(), 3),
    "-- most decliners keep declining rather than sit flat",
)

Base rates:
  was_declining (prior state): 0.238
  future_decline (of all pages): 0.396
  future_recovery (of already-declining pages, n=31997): 0.282
  future_momentum (of non-declining pages, n=102401): 0.248

Diagnostic -- stagnant (implicit negative class), split by prior state:
  stagnant, of non-declining pages: 0.232
  stagnant, of already-declining pages: 0.189 -- most decliners keep declining rather than sit flat


**Baseline-choice sanity check.** `future_change_pct` above compares the future 30 days against the most recent 30 days (the same window already used for the trend check). I first tried the full 90-day daily average as the baseline instead, and this cell is that actual test: a page's 90-day average can be inflated by an *older, no-longer-representative* period (e.g. a page that was healthy for 60 days then declined for the last 30) -- comparing recovery against that stale blended average hides real recoveries.

In [4]:
# Recompute the REJECTED 90-day-baseline version here, to compare against the
# 30-day baseline `df` already uses above (does not modify `df`).
prior_daily_90 = df["prior_impressions_90d_for_rejected_baseline_test"] / 90
future_daily = df["future_impressions"] / 30
future_change_pct_90 = (future_daily - prior_daily_90) / prior_daily_90 * 100

was_declining = df["was_declining"]
future_decline_90 = (~was_declining) & (future_change_pct_90 <= -20)
future_recovery_90 = (was_declining) & (future_change_pct_90 >= 20)
future_momentum_90 = (~was_declining) & (future_change_pct_90 >= 20)

print("future_recovery rate (of already-declining pages):")
print("  90-day baseline (rejected):    ", round(future_recovery_90[was_declining].mean(), 3))
print("  30-day baseline (adopted, current df):", round(df.loc[was_declining, "future_recovery"].mean(), 3))
print()
print("Agreement between the two baselines, per target:")
print("  future_decline: ", round((df["future_decline"] == future_decline_90).mean(), 3))
print("  future_recovery:", round((df.loc[was_declining, "future_recovery"] == future_recovery_90[was_declining]).mean(), 3))
print("  future_momentum:", round((df.loc[~was_declining, "future_momentum"] == future_momentum_90[~was_declining]).mean(), 3))

future_recovery rate (of already-declining pages):
  90-day baseline (rejected):     0.183
  30-day baseline (adopted, current df): 0.282

Agreement between the two baselines, per target:
  future_decline:  0.845
  future_recovery: 0.887


  future_momentum: 0.764


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**The core problem: a single threshold can't tell a real signal from noise.**

Back in `w01`, the starter CSV showed a `trend_pct` value as high as **44,900%** in the "up" bucket. That's not a typo — it happens when a page went from something tiny, like 1 impression, to a handful more. Going from 1 to 5 impressions is technically "+400%," but it's meaningless — that's random noise, not a real trend.

A rule like *"flag anything whose impressions changed by ±20%"* can't tell these two cases apart:
- a huge page that genuinely dropped 20% (a real signal, worth a specialist's time)
- a tiny page that went from 2 impressions to 1 (also "-50%," but just noise)

Both trip the same threshold. To fix that by hand, you'd have to keep bolting on more conditions ("...but only if it also has enough volume, and isn't too new, and..."), and it gets unmanageable fast. A model can weigh "% change" *together with* prior volume, age, and position all at once, and learn on its own that a 20% swing on a high-traffic page means something different than the same swing on a near-empty one — without a human hand-writing every combination.

**Even the definitions themselves needed more care than one number, twice over.** The window-choice sanity check found two reasonable window choices (30/30 vs 45/45) disagreeing on 21.9% of pages; the baseline-choice sanity check found the 90-day vs. 30-day future baseline disagreeing on `future_recovery` for over 11% of already-declining pages (18.3% → 28.2%). If just *defining the inputs* is this sensitive to small design choices, a single hand-written rule covering the whole decline/recovery/momentum picture is asking too much of one threshold.

**Real numbers from the actual data back this up:** 23.8% of pages were already declining at the sample decision point; of the rest, 24.8% went on to accelerate and 23.2% stayed flat; of the decliners, 28.2% recovered and 18.9% stayed flat rather than declining further. Three genuinely different, common populations that one fixed condition can't cleanly tell apart — and populations whose exact size depends on getting the window and baseline choices right, which is exactly why we tested rather than assumed both.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.